# Language Models and Text Generation

## What is a Language Model?
A language model assigns a probability to a sequence of words:

$$P(w_1, w_2, ..., w_T) = \prod_{t=1}^{T} P(w_t | w_1, ..., w_{t-1})$$

It answers: *"How likely is this sequence of words?"*

**Applications**: Text generation, machine translation, speech recognition, spell checking, autocomplete.

---

## 1. N-gram Language Models

Markov assumption only look at the last $n-1$ words:

$$P(w_t | w_1, ..., w_{t-1}) \approx P(w_t | w_{t-n+1}, ..., w_{t-1})$$

**Bigram model** (n=2):
$$P(w_t | w_{t-1}) = \frac{\text{count}(w_{t-1}, w_t)}{\text{count}(w_{t-1})}$$

### Perplexity
Measures how well the model predicts a held-out test set. **Lower is better.**

$$PP(W) = P(w_1, w_2, ..., w_N)^{-1/N} = \sqrt[N]{\frac{1}{P(w_1, ..., w_N)}}$$

Intuitively: the model is as confused as if it were choosing randomly among $PP(W)$ words.

### Smoothing
Handle zero-probability n-grams:
- **Laplace (Add-1)**: Add 1 to all counts
- **Add-k**: Add $k < 1$
- **Kneser-Ney**: Best smoothing, uses continuation probability

---

## 2. Decoding Strategies

Given a language model, how do we generate text?

### 2.1 Greedy Decoding
Always pick the most probable next token:
$$w_t = \arg\max_{w} P(w | w_{<t})$$
Fast but often repetitive and suboptimal.

### 2.2 Beam Search
Keep top-$k$ (beam width) partial sequences at each step.

Score: $\log P(w_1, ..., w_T) = \sum_{t=1}^{T} \log P(w_t | w_{<t})$

Length normalization: divide by $T^\alpha$ to avoid preference for short sequences.

### 2.3 Temperature Sampling
Scale logits by temperature $\tau$ before softmax:
$$P(w_i) = \frac{\exp(z_i / \tau)}{\sum_j \exp(z_j / \tau)}$$

- $\tau \to 0$: Greedy (deterministic)
- $\tau = 1$: Standard sampling
- $\tau > 1$: More random, creative

### 2.4 Top-k Sampling
Sample only from the top-$k$ most probable tokens:
$$P'(w) = \frac{P(w) \cdot \mathbf{1}[w \in \text{Top-}k]}{\sum_{w' \in \text{Top-}k} P(w')}$$

### 2.5 Top-p (Nucleus) Sampling
Sample from the smallest set whose cumulative probability $\geq p$:
$$V^{(p)} = \arg\min_V \sum_{w \in V} P(w) \geq p$$

Adapts dynamically: when distribution is flat, use more tokens; when peaked, use fewer.

---

## 3. Evaluation Metrics

### BLEU (Bilingual Evaluation Understudy)
Measures n-gram overlap between generated and reference text:

$$\text{BLEU} = BP \cdot \exp\left( \sum_{n=1}^{N} w_n \log p_n \right)$$

Where:
- $p_n$ = modified n-gram precision
- $w_n = 1/N$ (uniform weights)
- $BP$ = brevity penalty: $BP = e^{1 - r/c}$ if $c < r$ else $1$

### ROUGE (Recall-Oriented Understudy for Gisting Evaluation)
Used for summarization. Measures recall of n-grams:

$$\text{ROUGE-N} = \frac{\sum_{S \in \text{Refs}} \sum_{gram_n \in S} \text{Count}_{match}(gram_n)}{\sum_{S \in \text{Refs}} \sum_{gram_n \in S} \text{Count}(gram_n)}$$

### BERTScore
Uses contextual BERT embeddings to compute similarity:
$$\text{P} = \frac{1}{|\hat{x}|} \sum_{\hat{x}_j \in \hat{x}} \max_{x_i \in x} x_i^T \hat{x}_j$$

### METEOR, CIDEr, SPICE other generation metrics

---

## 4. Summarization

- **Extractive**: Select sentences from original text (TextRank, BertSum)
- **Abstractive**: Generate new text (BART, T5, Pegasus)

---

## 5. Machine Translation
Seq2Seq with attention, then Transformer (encoder-decoder). Training with teacher forcing.

In [1]:
import numpy as np
from collections import defaultdict, Counter
import math

# ============================================================
# N-GRAM LANGUAGE MODEL FROM SCRATCH
# ============================================================
class NgramLM:
    def __init__(self, n=2, smoothing_k=1.0):
        self.n = n
        self.k = smoothing_k  # Laplace/add-k smoothing
        self.ngrams = defaultdict(Counter)
        self.vocab = set()
    
    def train(self, corpus):
        """corpus: list of token lists"""
        for sentence in corpus:
            tokens = ['<s>'] * (self.n - 1) + sentence + ['</s>']
            self.vocab.update(tokens)
            for i in range(self.n - 1, len(tokens)):
                context = tuple(tokens[i - self.n + 1:i])
                word = tokens[i]
                self.ngrams[context][word] += 1
    
    def probability(self, word, context):
        """P(word | context) with add-k smoothing"""
        context = tuple(context[-(self.n-1):])
        count_context = sum(self.ngrams[context].values())
        count_word = self.ngrams[context][word]
        V = len(self.vocab)
        return (count_word + self.k) / (count_context + self.k * V)
    
    def perplexity(self, test_corpus):
        """Compute perplexity on test corpus"""
        log_prob = 0
        N = 0
        for sentence in test_corpus:
            tokens = ['<s>'] * (self.n - 1) + sentence + ['</s>']
            for i in range(self.n - 1, len(tokens)):
                context = tokens[i - self.n + 1:i]
                word = tokens[i]
                p = self.probability(word, context)
                log_prob += math.log(p + 1e-10)
                N += 1
        return math.exp(-log_prob / N)
    
    def generate(self, max_len=20, seed=None):
        """Generate text using the language model"""
        import random
        if seed: random.seed(seed)
        tokens = ['<s>'] * (self.n - 1)
        for _ in range(max_len):
            context = tuple(tokens[-(self.n-1):])
            if context not in self.ngrams:
                break
            candidates = list(self.ngrams[context].keys())
            probs = [self.probability(w, list(context)) for w in candidates]
            total = sum(probs)
            probs = [p/total for p in probs]
            next_word = random.choices(candidates, weights=probs)[0]
            if next_word == '</s>':
                break
            tokens.append(next_word)
        return ' '.join(tokens[self.n-1:])

# Train on sample corpus
corpus = [
    ["the", "cat", "sat", "on", "the", "mat"],
    ["the", "dog", "ran", "in", "the", "park"],
    ["the", "cat", "ran", "on", "the", "roof"],
    ["a", "dog", "sat", "by", "the", "door"],
    ["the", "cat", "played", "with", "the", "ball"],
]

lm = NgramLM(n=2, smoothing_k=0.1)
lm.train(corpus)

print(f"Bigram LM trained on {len(corpus)} sentences")
print(f"Vocabulary size: {len(lm.vocab)}")
print(f"\nP('cat' | 'the') = {lm.probability('cat', ['the']):.4f}")
print(f"P('dog' | 'the') = {lm.probability('dog', ['the']):.4f}")
print(f"\nPerplexity on training: {lm.perplexity(corpus):.4f}")
print("\nGenerated text:")
for i in range(3):
    print(f"  {lm.generate(seed=i+42)}")

Bigram LM trained on 5 sentences
Vocabulary size: 18

P('cat' | 'the') = 0.2870
P('dog' | 'the') = 0.1019

Perplexity on training: 3.4836

Generated text:
  the cat sat on the roof
  the roof
  the dog sat on the cat sat on the dog sat by the mat


In [2]:
# ============================================================
# DECODING STRATEGIES FROM SCRATCH
# ============================================================
import torch
import torch.nn.functional as F

def greedy_decode(logits):
    """Always pick most probable token"""
    return torch.argmax(logits, dim=-1)

def temperature_sample(logits, temperature=1.0):
    """Sample with temperature scaling"""
    scaled = logits / temperature
    probs = F.softmax(scaled, dim=-1)
    return torch.multinomial(probs, num_samples=1).squeeze()

def top_k_sample(logits, k=50, temperature=1.0):
    """Sample from top-k tokens only"""
    scaled = logits / temperature
    # Zero out all but top-k
    values, _ = torch.topk(scaled, k)
    min_val = values[-1]
    filtered = scaled.clone()
    filtered[filtered < min_val] = float('-inf')
    probs = F.softmax(filtered, dim=-1)
    return torch.multinomial(probs, num_samples=1).squeeze()

def top_p_sample(logits, p=0.9, temperature=1.0):
    """Nucleus sampling - sample from tokens whose cumsum >= p"""
    scaled = logits / temperature
    sorted_logits, sorted_indices = torch.sort(scaled, descending=True)
    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
    # Remove tokens with cumulative probability above threshold
    sorted_indices_to_remove = cumulative_probs - F.softmax(sorted_logits, dim=-1) >= p
    sorted_logits[sorted_indices_to_remove] = float('-inf')
    # Unsort
    logits_filtered = sorted_logits.gather(0, sorted_indices.argsort())
    probs = F.softmax(logits_filtered, dim=-1)
    return torch.multinomial(probs, num_samples=1).squeeze()

# Compare strategies on sample logits
torch.manual_seed(42)
vocab_size = 10
vocab = [f"word_{i}" for i in range(vocab_size)]
logits = torch.randn(vocab_size) * 2

probs = F.softmax(logits, dim=-1)
print("Token probabilities:")
for w, p in sorted(zip(vocab, probs.tolist()), key=lambda x: -x[1])[:5]:
    print(f"  {w}: {p:.4f}")

print(f"\nGreedy: {vocab[greedy_decode(logits).item()]}")
print(f"Temp=0.5: {vocab[temperature_sample(logits, 0.5).item()]}")
print(f"Temp=1.5: {vocab[temperature_sample(logits, 1.5).item()]}")
print(f"Top-k=3: {vocab[top_k_sample(logits, k=3).item()]}")
print(f"Top-p=0.9: {vocab[top_p_sample(logits, p=0.9).item()]}")

Token probabilities:
  word_6: 0.8758
  word_8: 0.0266
  word_0: 0.0207
  word_9: 0.0181
  word_2: 0.0169

Greedy: word_6
Temp=0.5: word_6
Temp=1.5: word_0
Top-k=3: word_6
Top-p=0.9: word_6


In [3]:
# ============================================================
# TEXT GENERATION WITH HUGGING FACE
# ============================================================
try:
    from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
    
    generator = pipeline('text-generation', model='gpt2')
    
    prompt = "The future of artificial intelligence"
    
    print(f"Prompt: '{prompt}'\n")
    
    # Greedy
    result = generator(prompt, max_new_tokens=30, do_sample=False, num_return_sequences=1)
    print(f"Greedy: {result[0]['generated_text']}")
    
    # Temperature sampling
    result = generator(prompt, max_new_tokens=30, do_sample=True, temperature=0.7, num_return_sequences=1)
    print(f"\nTemp=0.7: {result[0]['generated_text']}")
    
    # Top-p
    result = generator(prompt, max_new_tokens=30, do_sample=True, top_p=0.9, temperature=1.0, num_return_sequences=1)
    print(f"\nTop-p=0.9: {result[0]['generated_text']}")
    
    # Beam search
    result = generator(prompt, max_new_tokens=30, num_beams=5, early_stopping=True, num_return_sequences=1)
    print(f"\nBeam=5: {result[0]['generated_text']}")

except Exception as e:
    print(f"Install transformers and run: {e}")

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt: 'The future of artificial intelligence'



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


...

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'early_stopping', 'max_new_tokens', 'num_beams'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Top-p=0.9: The future of artificial intelligence is always coming. If we are to have a future in which all kinds of technologies replace human beings at the core of every facet of our lives,



Beam=5: The future of artificial intelligence is uncertain, but it's clear that the future of artificial intelligence is uncertain.

The future of artificial intelligence is uncertain, but it's clear


In [4]:
# ============================================================
# EVALUATION METRICS
# ============================================================
# pip install nltk rouge-score bert-score
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction

# BLEU Score
reference = ["the cat sat on the mat".split()]
hypothesis1 = "the cat sat on the mat".split()  # Perfect
hypothesis2 = "a cat is sitting on a rug".split()  # Different
hypothesis3 = "the cat".split()  # Too short

smoother = SmoothingFunction().method1
print("BLEU Scores:")
print(f"  Perfect match:  {sentence_bleu(reference, hypothesis1):.4f}")
print(f"  Similar:        {sentence_bleu(reference, hypothesis2, smoothing_function=smoother):.4f}")
print(f"  Short (BP penalty): {sentence_bleu(reference, hypothesis3, smoothing_function=smoother):.4f}")

# ROUGE
try:
    from rouge_score import rouge_scorer
    
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    reference_text = "The cat sat on the mat in the garden"
    hypothesis_text = "A cat was sitting on a mat outdoors"
    
    scores = scorer.score(reference_text, hypothesis_text)
    print("\nROUGE Scores:")
    for metric, score in scores.items():
        print(f"  {metric}: P={score.precision:.3f}, R={score.recall:.3f}, F1={score.fmeasure:.3f}")
except ImportError:
    print("\nInstall rouge-score: pip install rouge-score")

BLEU Scores:
  Perfect match:  1.0000
  Similar:        0.0393
  Short (BP penalty): 0.0428

ROUGE Scores:
  rouge1: P=0.375, R=0.333, F1=0.353
  rouge2: P=0.000, R=0.000, F1=0.000
  rougeL: P=0.375, R=0.333, F1=0.353


In [5]:
# ============================================================
# SUMMARIZATION WITH HUGGING FACE
# ============================================================
try:
    from transformers import pipeline
    
    summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
    
    article = """
    Artificial intelligence (AI) is transforming industries across the globe. 
    From healthcare to finance, AI systems are being deployed to automate tasks, 
    detect patterns, and make predictions. Machine learning, a subset of AI, 
    enables computers to learn from data without being explicitly programmed.
    Deep learning, using neural networks with many layers, has achieved 
    breakthrough results in image recognition, natural language processing, 
    and game playing. Companies like Google, Microsoft, and OpenAI are investing
    billions in AI research and development.
    """
    
    summary = summarizer(article, max_length=60, min_length=20, do_sample=False)
    print("Original length:", len(article.split()), "words")
    print("Summary:", summary[0]['summary_text'])
    print("Summary length:", len(summary[0]['summary_text'].split()), "words")
    
except Exception as e:
    print(f"Requires model download: {e}")

Requires model download: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"


## Additional Learning Resources

### Papers
- [BLEU: a Method for Automatic Evaluation of Machine Translation](https://aclanthology.org/P02-1040/) Papineni et al., 2002
- [ROUGE: A Package for Automatic Evaluation of Summaries](https://aclanthology.org/W04-1013/) Lin, 2004
- [BERTScore: Evaluating Text Generation with BERT](https://arxiv.org/abs/1904.09675) Zhang et al., 2020
- [BART: Denoising Sequence-to-Sequence Pre-training](https://arxiv.org/abs/1910.13461) Lewis et al., 2019
- [The Curious Case of Neural Text Degeneration (Nucleus Sampling)](https://arxiv.org/abs/1904.09751) Holtzman et al., 2020

### Textbooks
- [Speech and Language Processing Chapter 3: N-gram Language Models](https://web.stanford.edu/~jurafsky/slp3/) Jurafsky & Martin (free)

### Tutorials
- [Hugging Face Text Generation Guide](https://huggingface.co/docs/transformers/main_classes/text_generation)
- [How to Generate Text with Transformers](https://huggingface.co/blog/how-to-generate)
- [Hugging Face Summarization Task](https://huggingface.co/docs/transformers/tasks/summarization)